# Data exploration

Exploration of data structure and edge cases

## Config

In [1]:
import unicodedata
from pathlib import Path

import pandas as pd

DATA_DIR = Path("..") / "data" / "raw"
EXPORT_DATE = pd.Timestamp("2026-10-01")

## Helper functions

In [2]:
def load_raw_csv(path: Path, encoding: str, separator: str) -> pd.DataFrame:
    """Read a CSV as raw strings, keeping blanks as empty strings.

    Args:
        path: Path to the CSV file.
        encoding: File encoding, detected from the raw bytes.
        separator: Field separator.

    Returns:
        A DataFrame where every column has string dtype and no value is
        turned into NaN.
    """
    return pd.read_csv(
        path,
        dtype=str,
        keep_default_na=False,
        encoding=encoding,
        sep=separator,
    )

In [3]:
def value_shapes(values: pd.Series) -> pd.Series:
    """Summarise string formats by masking their characters.

    Digits become 9 and letters become a or A, so distinct formats (for
    example "9999-99-99" vs "9. 9. 9999") collapse into countable patterns.

    Args:
        values: A string column.

    Returns:
        Counts of each masked pattern, most frequent first.
    """
    digits_masked = values.str.replace(r"\d", "9", regex=True)
    lowercase_masked = digits_masked.str.replace(r"[a-záčďéěíňóřšťúůýž]", "a", regex=True)
    masked = lowercase_masked.str.replace(r"[A-ZÁČĎÉĚÍŇÓŘŠŤÚŮÝŽ]", "A", regex=True)
    return masked.value_counts()

In [4]:
def first_filled_value(column: pd.Series) -> str:
    """Return the first non-empty value of a column.

    Args:
        column: A raw string column.

    Returns:
        The first non-empty value, "" when every value is empty.
    """
    filled_values = column[column != ""]
    if filled_values.empty:
        return ""
    return filled_values.iloc[0]


def profile_columns(frame: pd.DataFrame) -> pd.DataFrame:
    """Describe each column by blanks, distinct values and a first example.

    Args:
        frame: Any raw string DataFrame.

    Returns:
        One row per column with the count of empty strings, the number of
        unique values and the first non-empty value seen.
    """
    is_empty_df = frame == ""
    empty_counts = is_empty_df.sum()
    unique_counts = frame.nunique()
    examples = frame.apply(first_filled_value)
    return pd.DataFrame({"empty": empty_counts, "unique": unique_counts, "example": examples})

In [5]:
def remove_diacritics(text: str) -> str:
    """Remove diacritics so that spellings with and without them compare equal.

    Args:
        text: Any text.

    Returns:
        The text without diacritics, e.g. 'Správa budov' -> 'Sprava budov'.
    """
    # NFKD splits a letter like "á" into "a" plus a separate accent mark.
    split_letters = unicodedata.normalize("NFKD", text)
    base_letters = [char for char in split_letters if not unicodedata.combining(char)]
    return "".join(base_letters)

In [6]:
def parse_mixed_dates(values: pd.Series) -> pd.Series:
    """Parse ISO and Czech dates (2019-01-06, 6. 1. 2019, 06.01.2019) for exploration.

    Args:
        values: Raw date strings.

    Returns:
        Parsed dates. Excel serial numbers and invalid values become NaT.
    """
    compact = values.str.replace(" ", "")
    iso_dates = pd.to_datetime(compact, format="%Y-%m-%d", errors="coerce")
    czech_dates = pd.to_datetime(compact, format="%d.%m.%Y", errors="coerce")
    return iso_dates.fillna(czech_dates)

In [7]:
def normalize_codes(codes: pd.Series) -> pd.Series:
    """Bring training codes to the catalog form, e.g. ' poz_och' -> 'POZ-OCH'.

    Args:
        codes: Raw training codes.

    Returns:
        Stripped, upper-case codes with underscores and spaces replaced by a dash.
    """
    stripped = codes.str.strip()
    upper = stripped.str.upper()
    return upper.str.replace(r"[_ ]+", "-", regex=True)

## 1. Loading

Encodings and separators were detected from the raw bytes

In [8]:
employees_df = load_raw_csv(DATA_DIR / "zamestnanci.csv", encoding="utf-8-sig", separator=",")
completed_df = load_raw_csv(DATA_DIR / "absolvovana_skoleni.csv", encoding="cp1250", separator=";")
catalog_df = load_raw_csv(DATA_DIR / "katalog_skoleni.csv", encoding="utf-8-sig", separator=",")
dates_df = load_raw_csv(DATA_DIR / "terminy_skoleni.csv", encoding="utf-8-sig", separator=";")

In [9]:
raw_frames = {
    "employees": employees_df,
    "completed": completed_df,
    "catalog": catalog_df,
    "dates": dates_df,
}
for name, frame in raw_frames.items():
    rows, columns = frame.shape
    column_list = ", ".join(frame.columns)
    print(f"{name:<10}{rows:>5} rows {columns:>3} columns")
    print(f"{'':<10}{column_list}")
    print()

employees   174 rows  12 columns
          id_zamestnance, jmeno, prijmeni, email, utvar, pozice, vedouci_email, datum_nastupu, datum_ukonceni, stav, rodne_cislo, poznamka

completed  2868 rows   9 columns
          id_zaznamu, id_zamestnance, zamestnanec, kod_skoleni, datum_absolvovani, platnost_do, poskytovatel, cena_kc, poznamka

catalog      10 rows   5 columns
          kod, nazev, platnost_mesicu, povinne_pro, poznamka

dates        34 rows   9 columns
          id_terminu, kod_skoleni, datum, cas, misto, kapacita, obsazeno, cena_na_osobu, stav



## 2. Catalog (`katalog_skoleni.csv`)


In [10]:
catalog_df

,kod,nazev,platnost_mesicu,povinne_pro,poznamka
0,BOZP-Z,BOZP – základní školení,24,VSICHNI,
1,POZ-OCH,Požární ochrana,24,VSICHNI,
2,GDPR,Ochrana osobních údajů,36,VSICHNI,
3,KYBER,Kybernetická bezpečnost,12,VSICHNI,
4,PRV-POM,První pomoc,36,UTVAR:Laboratoře;Správa budov;Knihovna,
5,RIDIC,Řidiči referenti,12,POZICE:Řidič,
6,LAB-BEZ,Bezpečná práce v laboratoři,24,UTVAR:Laboratoře,
7,ZDVIH,Obsluha zdvihacích zařízení,2 roky,POZICE:Technik údržby,
8,LEAD,Základy vedení lidí,0,POZICE:Vedoucí,
9,ANGL,Anglický jazyk – kurz,,NEPOVINNE,


## 3. Employees (`zamestnanci.csv`)

`rodne_cislo` - personal data, therefore dropped immediately after loading.

In [11]:
has_birth_number = employees_df["rodne_cislo"] != ""
birth_number_count = has_birth_number.sum()
print(f"rodne_cislo non-empty: {birth_number_count} of {len(employees_df)}")
employees_df = employees_df.drop(columns="rodne_cislo")

rodne_cislo non-empty: 174 of 174


In [12]:
contact_columns = ["email", "vedouci_email"]
no_contact_employees_df = employees_df.drop(columns=contact_columns)
profile_columns(no_contact_employees_df)

,empty,unique,example
id_zamestnance,0,174,E0001
jmeno,0,65,Tereza
prijmeni,0,103,Urbanová
utvar,5,18,Studijní oddělení
pozice,0,11,Vedoucí
datum_nastupu,0,166,2008-02-23
datum_ukonceni,158,17,2027-01-20
stav,0,3,aktivní
poznamka,170,5,-2+3


Detecting unexpected or inconsistent category values: unknown departments or positions, casing and spelling variants, and whether `stav` is drawn from a small fixed set.

In [13]:
employees_df["stav"].value_counts(dropna=False)

stav
aktivní       156
ukončen        12
rodičovská      6
Name: count, dtype: int64

In [14]:
employees_df["utvar"].value_counts(dropna=False)

utvar
Laboratoře             31
Správa budov           27
IT oddělení            22
Knihovna               21
Studijní oddělení      20
Ekonomické oddělení    19
Personální oddělení     8
Marketing               6
                        5
Vedení                  5
Laboratore              3
Studijní                1
Studijni oddeleni       1
Sprava budov            1
SPRÁVA BUDOV            1
správa budov            1
Personalni oddeleni     1
VEDENÍ                  1
Name: count, dtype: int64

In [15]:
employees_df["pozice"].value_counts(dropna=False)

pozice
Laborant          28
Specialista       24
Referent          24
Technik           20
Technik údržby    16
Knihovník         15
Vedoucí           13
Účetní            12
Asistent          10
Řidič              8
Vývojář            4
Name: count, dtype: int64

Detecting which date formats appear in the employee dates, and whether more than one format (or an invalid value) is mixed in one column.

In [16]:
value_shapes(employees_df["datum_nastupu"])

datum_nastupu
9999-99-99     169
9. 9. 9999       3
99. 9. 9999      2
Name: count, dtype: int64

In [17]:
value_shapes(employees_df["datum_ukonceni"])

datum_ukonceni
              158
9999-99-99     16
Name: count, dtype: int64

Detecting duplicate employee ids or emails, and whether every id follows one consistent pattern.

In [18]:
is_duplicate_id = employees_df["id_zamestnance"].duplicated()
duplicate_id_count = is_duplicate_id.sum()
print(f"duplicate id_zamestnance: {duplicate_id_count}")

duplicate id_zamestnance: 0


In [19]:
emails = employees_df["email"].str.lower()
is_duplicate_email = emails.duplicated()
duplicate_email_count = is_duplicate_email.sum()
print(f"duplicate email: {duplicate_email_count}")

duplicate email: 2


Checking whether the id of employees follows one consistent pattern

In [20]:
value_shapes(employees_df["id_zamestnance"])

id_zamestnance
A9999    174
Name: count, dtype: int64

Detecting the same person under two ids: same first name, department, position, start date and manager.

In [21]:
same_person_columns = ["jmeno", "utvar", "pozice", "datum_nastupu", "vedouci_email"]
is_same_person = employees_df.duplicated(same_person_columns, keep=False)
shown_columns = ["id_zamestnance", "prijmeni", "jmeno", "utvar", "pozice", "datum_nastupu"]
same_person_df = employees_df.loc[is_same_person, shown_columns]
same_person_df.sort_values(["jmeno", "datum_nastupu", "id_zamestnance"])

,id_zamestnance,prijmeni,jmeno,utvar,pozice,datum_nastupu
154,E0155,Novotný,Adam,Personální oddělení,Referent,2018-04-21
170,E1000,Novotny,Adam,Personální oddělení,Referent,2018-04-21
41,E0042,Zdražil,Jaroslav,Ekonomické oddělení,Účetní,2016-12-26
171,E1001,Zdražiy,Jaroslav,Ekonomické oddělení,Účetní,2016-12-26
69,E0070,Hruška,Petr,Knihovna,Knihovník,2022-08-07
173,E1003,Hrušky,Petr,Knihovna,Knihovník,2022-08-07
123,E0124,Havlíková,Petra,Laboratoře,Laborant,2015-09-01
172,E1002,Havlikova,Petra,Laboratoře,Laborant,2015-09-01


Detecting malformed emails: embedded spaces, a comma instead of a dot, or a missing domain.

In [22]:
is_wellformed_email = employees_df["email"].str.fullmatch(r"[^@\s,]+@[^@\s,]+\.[a-z]+")
is_malformed_email = ~is_wellformed_email
malformed_email_count = is_malformed_email.sum()
print(f"malformed email: {malformed_email_count}")

malformed email: 6


In [23]:
malformed_email_ids = employees_df.loc[is_malformed_email, "id_zamestnance"]
malformed_email_ids.tolist()

['E0002', 'E0099', 'E0121', 'E0129', 'E0150', 'E0157']

Detecting leading or trailing spaces in names, which break exact name comparisons.

In [24]:
stripped_first_name = employees_df["jmeno"].str.strip()
has_outer_spaces_first_name = employees_df["jmeno"] != stripped_first_name
outer_spaces_first_name_count = has_outer_spaces_first_name.sum()
print(f"jmeno with outer spaces: {outer_spaces_first_name_count}")

jmeno with outer spaces: 8

In [25]:
stripped_last_name = employees_df["prijmeni"].str.strip()
has_outer_spaces_last_name = employees_df["prijmeni"] != stripped_last_name
outer_spaces_last_name_count = has_outer_spaces_last_name.sum()
print(f"prijmeni with outer spaces: {outer_spaces_last_name_count}")

prijmeni with outer spaces: 0


Detecting end dates that disagree with status: active people that still carry an end date, or terminated people without one.

In [26]:
has_end_date = employees_df["datum_ukonceni"] != ""
pd.crosstab(employees_df["stav"], has_end_date)

datum_ukonceni,False,True
stav,,
aktivní,152,4
rodičovská,6,0
ukončen,0,12


Detecting manager references that point to no known employee: orphaned or placeholder `vedouci_email` values.

In [27]:
employee_emails = employees_df["email"].str.lower()
manager_emails = employees_df["vedouci_email"].str.lower()
has_manager = manager_emails != ""
is_known_manager = manager_emails.isin(employee_emails)
is_unknown_manager = has_manager & ~is_known_manager
unknown_manager_count = is_unknown_manager.sum()
print(f"vedouci_email not found among employees: {unknown_manager_count}")

vedouci_email not found among employees: 3


In [28]:
unknown_manager_ids = employees_df.loc[is_unknown_manager, "id_zamestnance"]
unknown_manager_ids.tolist()

['E0023', 'E0080', 'E0097']

In [29]:
unknown_manager_emails = manager_emails[is_unknown_manager]
unknown_manager_emails.value_counts()

vedouci_email
nezname.jmeno@vzorova.example    3
Name: count, dtype: int64

## 4. Completed trainings (`absolvovana_skoleni.csv`)

First rows and a profile of every column: empty values, distinct values and a first example.

In [30]:
completed_df.head(10)

,id_zaznamu,id_zamestnance,zamestnanec,kod_skoleni,datum_absolvovani,platnost_do,poskytovatel,cena_kc,poznamka
0,Z000001,E0131,Vacková Hana,BOZP-Z,1900-01-01,1902-01-01,Interní školitel,"0,00",
1,Z000002,E0149,Navrátil Miroslav,BOZP-Z,1. 1. 1900,1. 1. 1902,Interní školitel,"0,00",
2,Z000003,E0024,Blažek Adam,BOZP-Z,2019-01-06,2021-01-06,Interní školitel,"450,00",
3,Z000004,E0063,Nováková Klára,GDPR,6. 1. 2019,6. 1. 2022,Vzorový školicí ústav,"700,00",
4,Z000005,E0088,Matoušková Adéla,POZ-OCH,6. 1. 2019,6. 1. 2021,E-learning portál,"380,00",online verze
5,Z000006,E0096,Novák Daniel,POZ-OCH,6. 1. 2019,6. 1. 2021,Vzorový školicí ústav,"0,00",
6,Z000007,E0118,Konečný Pavel,BOZP-Z,6. 1. 2019,6. 1. 2021,Vzorový školicí ústav,"550,00",
7,Z000008,E0067,Poláková Kateřina,BOZP-Z,7. 1. 2019,7. 1. 2021,Externí lektor,"550,00",absolvováno v zahraničí
8,Z000009,E0094,Pospíšil Vladimír,GDPR,07.01.2019,07.01.2024,Vzorový školicí ústav,"550,00",
9,Z000010,E0103,Prokopová Lenka,GDPR,7. 1. 2019,7. 1. 2022,E-learning portál,"650,00",


In [31]:
profile_columns(completed_df)

,empty,unique,example
id_zaznamu,0,2853,Z000001
id_zamestnance,2,182,E0131
zamestnanec,2,174,Vacková Hana
kod_skoleni,2,31,BOZP-Z
datum_absolvovani,2,1985,1900-01-01
platnost_do,49,2057,1902-01-01
poskytovatel,2,10,Interní školitel
cena_kc,21,47,"0,00"
poznamka,2599,14,online verze


Detecting the mix of formats in the completion and validity dates (ISO, `d. m. Y`, and any values that are not dates at all).

In [32]:
value_shapes(completed_df["datum_absolvovani"])

datum_absolvovani
99. 9. 9999          968
9999-99-99           705
99.99.9999           462
9. 9. 9999           397
99. 99. 9999         242
9. 99. 9999           88
99999                  3
                       2
aaaaa_aaaaaaaaaaa      1
Name: count, dtype: int64

In [33]:
value_shapes(completed_df["platnost_do"])

platnost_do
99. 9. 9999     954
9999-99-99      692
99.99.9999      449
9. 9. 9999      390
99. 99. 9999    243
9. 99. 9999      87
                 49
99999             3
aaaaaaaa_aa       1
Name: count, dtype: int64

Detecting training codes that match no catalog code: typos, casing or spacing variants, and genuinely unknown codes.

In [34]:
is_catalog_code = completed_df["kod_skoleni"].isin(catalog_df["kod"])
unknown_codes = completed_df.loc[~is_catalog_code, "kod_skoleni"]
print("kod_skoleni not in catalog:")
unknown_codes.value_counts()

kod_skoleni not in catalog:


kod_skoleni
KYBER          6
BOZP-X         4
 BOZP-Z        2
BOZP-Z         2
POZ_OCH        2
               2
 POZ-OCH       2
POZ-OCH        2
POZ OCH        2
lead           1
BOZP Z         1
 KYBER         1
bozp-z         1
 GDPR          1
GDPR           1
kod_skoleni    1
GDPR-2         1
kyber          1
BOZP_Z         1
LAB-BEZ        1
KYB3R          1
Name: count, dtype: int64

Most codes above are catalog codes written differently: outer spaces (`'KYBER '` shows as `KYBER`), lowercase, or `_` and spaces instead of `-`. Detecting what is left after stripping, upper-casing and unifying separators. The empty value and `KOD-SKOLENI` come from junk rows (empty rows, repeated header).

In [35]:
normalized_codes = normalize_codes(completed_df["kod_skoleni"])
is_still_unknown = ~normalized_codes.isin(catalog_df["kod"])
still_unknown_codes = normalized_codes[is_still_unknown]
print("kod_skoleni not in catalog after normalising:")
still_unknown_codes.value_counts()

kod_skoleni not in catalog after normalising:


kod_skoleni
BOZP-X         4
               2
KOD-SKOLENI    1
GDPR-2         1
KYB3R          1
Name: count, dtype: int64

Detecting completions linked to an employee id that is absent from the employee table.

In [36]:
is_known_employee = completed_df["id_zamestnance"].isin(employees_df["id_zamestnance"])
unknown_employee_ids = completed_df.loc[~is_known_employee, "id_zamestnance"]
print("id_zamestnance not in employees:")
unknown_employee_ids.value_counts()

id_zamestnance not in employees:


id_zamestnance
                  2
E9006             1
E9001             1
id_zamestnance    1
E9008             1
E9009             1
E9007             1
E9004             1
E9002             1
E9003             1
E9005             1
Name: count, dtype: int64

Counting records per training code and per provider. Code variants show up as separate rows; provider names come in mixed case (the upper-case ones are the copies found below).

In [37]:
completed_df["kod_skoleni"].value_counts()

kod_skoleni
KYBER          838
BOZP-Z         573
POZ-OCH        528
GDPR           436
PRV-POM        212
LAB-BEZ        114
RIDIC           51
ZDVIH           46
ANGL            24
LEAD            10
KYBER            6
BOZP-X           4
 BOZP-Z          2
BOZP-Z           2
POZ_OCH          2
                 2
 POZ-OCH         2
POZ-OCH          2
POZ OCH          2
lead             1
BOZP Z           1
 KYBER           1
bozp-z           1
 GDPR            1
GDPR             1
kod_skoleni      1
GDPR-2           1
kyber            1
BOZP_Z           1
LAB-BEZ          1
KYB3R            1
Name: count, dtype: int64

In [38]:
completed_df["poskytovatel"].value_counts()

poskytovatel
E-learning portál        721
Vzorový školicí ústav    720
Externí lektor           713
Interní školitel         701
VZOROVÝ ŠKOLICÍ ÚSTAV      3
E-LEARNING PORTÁL          3
                           2
INTERNÍ ŠKOLITEL           2
EXTERNÍ LEKTOR             2
poskytovatel               1
Name: count, dtype: int64

Detecting the number format used for prices (decimal comma, thousands separators) so the cleaning step can parse it.

In [39]:
value_shapes(completed_df["cena_kc"])

cena_kc
999,99      1685
9,99         844
9 999,99     303
              21
999,- Aa       4
aaaaaa         4
-999,99        3
9,- Aa         2
aaaa_aa        1
9999           1
Name: count, dtype: int64

Detecting duplicate records: the same id written twice, and copies under another id. Dates are parsed and codes normalised first, so a copy in another date format is still caught.

In [40]:
is_record_id = completed_df["id_zaznamu"].str.fullmatch(r"Z\d{6}")
valid_records_df = completed_df[is_record_id]
is_repeated_id = valid_records_df["id_zaznamu"].duplicated()
repeated_id_count = is_repeated_id.sum()
print(f"repeated id_zaznamu: {repeated_id_count}")

repeated id_zaznamu: 15


In [41]:
unique_records_df = valid_records_df.drop_duplicates("id_zaznamu")
unique_records_df["completed_on"] = parse_mixed_dates(unique_records_df["datum_absolvovani"])
unique_records_df["code"] = normalize_codes(unique_records_df["kod_skoleni"])

In [42]:
key_columns = ["id_zamestnance", "code", "completed_on"]
is_copy = unique_records_df.duplicated(key_columns)
copy_count = is_copy.sum()
print(f"copies under another id: {copy_count}")

copies under another id: 10


In [43]:
is_in_copy_pair = unique_records_df.duplicated(key_columns, keep=False)
shown_columns = ["id_zaznamu", *key_columns, "poskytovatel"]
copy_pairs_df = unique_records_df.loc[is_in_copy_pair, shown_columns]
copy_pairs_df.sort_values(key_columns)

,id_zaznamu,id_zamestnance,code,completed_on,poskytovatel
899,Z000895,E0017,BOZP-Z,2021-05-31,E-learning portál
2865,Z988277,E0017,BOZP-Z,2021-05-31,E-LEARNING PORTÁL
2582,Z002567,E0019,POZ-OCH,2025-08-29,Interní školitel
2860,Z951895,E0019,POZ-OCH,2025-08-29,INTERNÍ ŠKOLITEL
1469,Z001462,E0028,BOZP-Z,2023-03-20,Vzorový školicí ústav
2857,Z915067,E0028,BOZP-Z,2023-03-20,VZOROVÝ ŠKOLICÍ ÚSTAV
1097,Z001090,E0060,POZ-OCH,2022-02-27,Vzorový školicí ústav
2862,Z982602,E0060,POZ-OCH,2022-02-27,VZOROVÝ ŠKOLICÍ ÚSTAV
1545,Z001538,E0062,BOZP-Z,2023-04-25,E-learning portál
2861,Z960898,E0062,BOZP-Z,2023-04-25,E-LEARNING PORTÁL


Splitting the name mismatches by cause: extra spaces, letter case, dropped diacritics, and what is left as a real mismatch.

In [44]:
full_names = employees_df["prijmeni"] + " " + employees_df["jmeno"]
full_name_by_id = full_names.set_axis(employees_df["id_zamestnance"])
named_df = unique_records_df.copy()
named_df["expected"] = named_df["id_zamestnance"].map(full_name_by_id)

In [45]:
has_known_employee = named_df["expected"].notna()
is_name_different = named_df["zamestnanec"] != named_df["expected"]
is_mismatch = has_known_employee & is_name_different
mismatch_df = named_df[is_mismatch]
mismatch_count = len(mismatch_df)
print(f"name differs from the employee record: {mismatch_count}")

name differs from the employee record: 121


In [46]:
actual_words = mismatch_df["zamestnanec"].str.split()
expected_words = mismatch_df["expected"].str.split()
actual = actual_words.str.join(" ")
expected = expected_words.str.join(" ")
equal_ignoring_spaces = actual == expected
spaces_count = equal_ignoring_spaces.sum()
print(f"extra spaces only: {spaces_count}")

extra spaces only: 21


In [47]:
actual_lower = actual.str.lower()
expected_lower = expected.str.lower()
equal_ignoring_case = actual_lower == expected_lower
differs_in_case_only = equal_ignoring_case & ~equal_ignoring_spaces
case_count = differs_in_case_only.sum()
print(f"letter case only: {case_count}")

letter case only: 95


In [48]:
actual_plain = actual_lower.map(remove_diacritics)
expected_plain = expected_lower.map(remove_diacritics)
equal_ignoring_diacritics = actual_plain == expected_plain
differs_in_diacritics_only = equal_ignoring_diacritics & ~equal_ignoring_case
diacritics_count = differs_in_diacritics_only.sum()
print(f"diacritics only: {diacritics_count}")

diacritics only: 0


In [49]:
is_real_mismatch = ~equal_ignoring_diacritics
real_mismatch_count = is_real_mismatch.sum()
print(f"real mismatch: {real_mismatch_count}")
mismatch_df.loc[is_real_mismatch, ["id_zaznamu", "id_zamestnance"]]

real mismatch: 5


,id_zaznamu,id_zamestnance
564,Z000561,E0029
595,Z000592,E0143
842,Z000838,E0094
1378,Z001371,E0068
1571,Z001564,E0052


Detecting completions dated before the employee was hired, such as the `1900-01-01` placeholder.

In [50]:
raw_hire_dates = employees_df.set_index("id_zamestnance")["datum_nastupu"]
hire_dates = parse_mixed_dates(raw_hire_dates)
employee_hired_on = unique_records_df["id_zamestnance"].map(hire_dates)
is_before_hire = unique_records_df["completed_on"] < employee_hired_on
before_hire_count = is_before_hire.sum()
print(f"completed before the employee was hired: {before_hire_count}")
unique_records_df.loc[is_before_hire, ["id_zaznamu", "id_zamestnance", "datum_absolvovani"]]

completed before the employee was hired: 2


,id_zaznamu,id_zamestnance,datum_absolvovani
0,Z000001,E0131,1900-01-01
1,Z000002,E0149,1. 1. 1900


Parsing record dates and codes once for the next checks. Junk rows are skipped; the 3 Excel serial dates stay unparsed.

In [51]:
records_df = completed_df[is_record_id]
records_df["completed_on"] = parse_mixed_dates(records_df["datum_absolvovani"])
records_df["valid_until"] = parse_mixed_dates(records_df["platnost_do"])
records_df["code"] = normalize_codes(records_df["kod_skoleni"])

Detecting completions dated after the export date, which cannot be finished trainings.

In [52]:
is_future = records_df["completed_on"] > EXPORT_DATE
future_count = is_future.sum()
print(f"dated after export: {future_count}")
records_df.loc[is_future, ["id_zaznamu", "datum_absolvovani", "poznamka"]]

dated after export: 5


,id_zaznamu,datum_absolvovani,poznamka
2852,Z002836,27.10.2026,zadáno předem
2853,Z002837,25.11.2026,zadáno předem
2854,Z002838,28. 11. 2026,zadáno předem
2855,Z002839,4. 12. 2026,zadáno předem
2856,Z002840,2026-12-31,zadáno předem


Detecting `platnost_do` values that disagree with completion date + catalog validity.

In [53]:
raw_validity = catalog_df.set_index("kod")["platnost_mesicu"]
raw_validity = raw_validity.replace({"2 roky": "24"})
validity_months = pd.to_numeric(raw_validity, errors="coerce")
records_df["catalog_months"] = records_df["code"].map(validity_months)

In [54]:
has_catalog_validity = records_df["catalog_months"] > 0
has_completion_date = records_df["completed_on"].notna()
has_validity_date = records_df["valid_until"].notna()
is_comparable = has_catalog_validity & has_completion_date & has_validity_date
comparable_df = records_df[is_comparable]
# DateOffset takes a single month count, not a column, so months are added row by row.
comparable_df["expected_until"] = [
    completed_on + pd.DateOffset(months=int(months))
    for completed_on, months in zip(comparable_df["completed_on"], comparable_df["catalog_months"])
]

In [55]:
is_different = comparable_df["valid_until"] != comparable_df["expected_until"]
different_count = is_different.sum()
print(f"comparable records: {len(comparable_df)}, platnost_do differs: {different_count}")

comparable records: 2809, platnost_do differs: 33


In [56]:
different_by_code = is_different.groupby(comparable_df["code"]).sum()
different_by_code

code
BOZP-Z     5
GDPR       8
KYBER      7
LAB-BEZ    0
POZ-OCH    8
PRV-POM    4
RIDIC      1
ZDVIH      0
dtype: int64

How far off the disagreeing `platnost_do` values are, in months.

In [57]:
DAYS_PER_MONTH = 30.44
time_off = comparable_df["valid_until"] - comparable_df["expected_until"]
days_off = time_off.dt.days
months_off = days_off[is_different] / DAYS_PER_MONTH
months_off = months_off.round()
months_off.value_counts()

-6.0     11
 24.0     8
 12.0     4
-12.0     2
-15.0     2
-29.0     1
-30.0     1
-41.0     1
-28.0     1
-26.0     1
-14.0     1
Name: count, dtype: int64

Detecting empty `platnost_do` per training. ANGL and LEAD have no catalog validity, so they are left out.

In [58]:
stripped_validity = records_df["platnost_do"].str.strip()
is_empty_validity = stripped_validity == ""
# has_catalog_validity comes from the platnost_do check above.
is_missing_validity = is_empty_validity & has_catalog_validity
missing_validity_codes = records_df.loc[is_missing_validity, "code"]
missing_validity_codes.value_counts()

code
POZ-OCH    4
BOZP-Z     3
KYBER      3
PRV-POM    2
Name: count, dtype: int64

## 5. Training dates (`terminy_skoleni.csv`)

Small enough to read in full.

In [59]:
dates_df

,id_terminu,kod_skoleni,datum,cas,misto,kapacita,obsazeno,cena_na_osobu,stav
0,T001,KYBER,2026-10-14,09:00-12:00,Aula A,25,22,800,plánováno
1,T002,KYBER,2026-10-28,09:00-12:00,Aula A,40,0,800,plánováno
2,T003,KYBER,2026-11-11,09:00-12:00,Aula A,30,5,800,plánováno
3,T004,KYBER,2026-11-25,09:00-12:00,Aula A,30,0,800,plánováno
4,T005,KYBER,2026-12-09,09:00-12:00,Aula A,25,0,800,plánováno
5,T006,KYBER,2026-12-16,13:00-15:00,Online,50,0,800,plánováno
6,T007,KYBER,2026-11-14,09:00-12:00,Online,60,0,800,plánováno
7,T008,KYBER,2026-09-23,09:00-12:00,Aula A,30,0,800,plánováno
8,T009,KYBER,2026-11-11,09:00-12:00,Aula A,30,5,800,plánováno
9,T010,KYB3R,2026-12-02,09:00-12:00,Aula B,20,0,800,plánováno


In [60]:
profile_columns(dates_df)

,empty,unique,example
id_terminu,0,34,T001
kod_skoleni,0,10,KYBER
datum,0,31,2026-10-14
cas,0,10,09:00-12:00
misto,0,9,Aula A
kapacita,0,11,25
obsazeno,0,13,22
cena_na_osobu,2,11,800
stav,0,2,plánováno


Detecting the status and location vocabularies, any term whose training code is not in the catalog, duplicate term ids, and the date format.

In [61]:
dates_df["stav"].value_counts()

stav
plánováno    32
zrušeno       2
Name: count, dtype: int64

In [62]:
dates_df["misto"].value_counts()

misto
Aula A                12
Online                 5
Učebna 1               4
Učebna 2               4
Autoškola Vzor         3
Laboratoř 3            2
Dvůr – sklad           2
Aula B                 1
Seminární místnost     1
Name: count, dtype: int64

In [63]:
dates_df["cas"].value_counts()

cas
09:00-12:00    9
08:00-11:00    5
10:00-12:00    4
09:00-15:00    4
14:00-16:00    3
08:00-12:00    3
09:00-13:00    2
07:30-15:30    2
13:00-15:00    1
09:00-17:00    1
Name: count, dtype: int64

In [64]:
is_unknown_term_code = ~dates_df["kod_skoleni"].isin(catalog_df["kod"])
unknown_term_code_count = is_unknown_term_code.sum()
print(f"kod_skoleni not in catalog: {unknown_term_code_count}")
dates_df.loc[is_unknown_term_code, ["id_terminu", "kod_skoleni"]]

kod_skoleni not in catalog: 1


,id_terminu,kod_skoleni
9,T010,KYB3R


In [65]:
is_duplicate_term_id = dates_df["id_terminu"].duplicated()
duplicate_term_id_count = is_duplicate_term_id.sum()
print(f"duplicate id_terminu: {duplicate_term_id_count}")

duplicate id_terminu: 0


In [66]:
value_shapes(dates_df["datum"])

datum
9999-99-99     33
99. 9. 9999     1
Name: count, dtype: int64

Detecting non-numeric values in the numeric columns, and any term booked beyond its capacity (`obsazeno > kapacita`).

In [67]:
number_columns = ["kapacita", "obsazeno", "cena_na_osobu"]
raw_numbers_df = dates_df[number_columns]
numbers_df = raw_numbers_df.apply(pd.to_numeric, errors="coerce")
is_non_numeric_df = numbers_df.isna()
is_non_numeric_df.sum()

kapacita         0
obsazeno         0
cena_na_osobu    3
dtype: int64

In [68]:
is_overbooked = numbers_df["obsazeno"] > numbers_df["kapacita"]
dates_df[is_overbooked]

,id_terminu,kod_skoleni,datum,cas,misto,kapacita,obsazeno,cena_na_osobu,stav
15,T016,BOZP-Z,2026-11-04,08:00-11:00,Aula A,30,31,450,plánováno
29,T030,LAB-BEZ,2026-10-29,09:00-13:00,Laboratoř 3,20,21,1500,plánováno


Detecting which terms fall before or after the export date, to separate usable future terms from ones already in the past.

In [69]:
parsed_dates = pd.to_datetime(dates_df["datum"], format="%Y-%m-%d", errors="coerce")
time_from_export = parsed_dates - EXPORT_DATE
days_from_export = time_from_export.dt.days
timeline_df = dates_df.assign(days_from_export=days_from_export)
timeline_df = timeline_df[["id_terminu", "kod_skoleni", "datum", "stav", "days_from_export"]]
timeline_df = timeline_df.sort_values("days_from_export")
timeline_df.head(15)

,id_terminu,kod_skoleni,datum,stav,days_from_export
18,T019,BOZP-Z,2026-09-16,plánováno,-15.0
7,T008,KYBER,2026-09-23,plánováno,-8.0
21,T022,GDPR,2026-09-30,plánováno,-1.0
14,T015,BOZP-Z,2026-10-07,plánováno,6.0
0,T001,KYBER,2026-10-14,plánováno,13.0
26,T027,RIDIC,2026-10-15,plánováno,14.0
10,T011,POZ-OCH,2026-10-20,plánováno,19.0
22,T023,PRV-POM,2026-10-22,plánováno,21.0
1,T002,KYBER,2026-10-28,plánováno,27.0
29,T030,LAB-BEZ,2026-10-29,plánováno,28.0


## 6. Insights

Insights and decisions are listed in `01_cleaning_ntb.ipynb`.